# LLaDA: Large Language Diffusion Models Algorithm

This notebook contains the actual implementation of the LLaDA (Large Language Diffusion with mAsking) algorithm as described in the paper: [Large Language Diffusion Models](https://arxiv.org/abs/2502.09992).

LLaDA is a diffusion model trained from scratch that challenges the notion that autoregressive models are the cornerstone of large language models. It uses a forward data masking process and a reverse process parameterized by a vanilla Transformer to predict masked tokens.

## Key Features:
- **Diffusion-based language modeling** instead of autoregressive generation
- **Masked diffusion process** with forward and reverse processes
- **Transformer encoder architecture** (removes causal mask from self-attention)
- **Semi-autoregressive sampling** with remasking strategies
- **Classifier-free guidance** for improved generation quality


## 1. Core Algorithm Components

The LLaDA algorithm consists of several key components:

1. **Forward Process**: Adds noise by masking tokens with a linear noise schedule
2. **Reverse Process**: Denoises by predicting masked tokens using a Transformer encoder
3. **Sampling Process**: Generates text through iterative denoising with remasking
4. **Classifier-Free Guidance**: Improves generation quality through guidance

Let's implement each component step by step.


In [ ]:
# Google Colab Setup - Device Detection and Memory Management
import torch
import torch.nn.functional as F
import numpy as np
from typing import Optional, Tuple, Union, Dict
import math
import gc

# Detect and set device properly for Google Colab
def setup_device():
    """Setup device for Google Colab with proper GPU/TPU detection"""
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
        print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device('mps')
        print("✅ Using Apple Silicon GPU (MPS)")
    else:
        device = torch.device('cpu')
        print("⚠️  Using CPU (consider enabling GPU in Colab: Runtime > Change runtime type > GPU)")

    return device

# Setup device
device = setup_device()

# Constants
MASK_TOKEN_ID = 126336  # The token ID of [MASK] in LLaDA
VOCAB_SIZE = 128256  # Typical vocabulary size for LLaDA

# Token-to-Interval Mapping
class TokenIntervalMapper:
    """
    Maps discrete token IDs to continuous intervals in [0, 1).
    Each token gets a unique interval, and all intervals are disjoint.
    """
    def __init__(self, vocab_size: int = VOCAB_SIZE):
        self.vocab_size = vocab_size
        self.interval_width = 1.0 / vocab_size
        self.token_to_interval = {}
        
        # Create mapping for each token
        for token_id in range(vocab_size):
            start = token_id * self.interval_width
            end = (token_id + 1) * self.interval_width
            self.token_to_interval[token_id] = (start, end)
    
    def token_to_interval(self, token_id: int) -> Tuple[float, float]:
        """Convert token ID to its corresponding interval [start, end)"""
        return self.token_to_interval[token_id]
    
    def token_to_center(self, token_id: int) -> float:
        """Convert token ID to the center of its interval"""
        start, end = self.token_to_interval[token_id]
        return (start + end) / 2
    
    def interval_to_token(self, value: float) -> int:
        """Convert a continuous value to the corresponding token ID"""
        if value < 0 or value >= 1:
            raise ValueError(f"Value {value} must be in [0, 1)")
        return int(value * self.vocab_size)
    
    def token_to_tensor(self, token_id: int) -> torch.Tensor:
        """Convert token ID to its continuous representation tensor"""
        center = self.token_to_center(token_id)
        return torch.tensor(center, dtype=torch.float32)
    
    def tokens_to_tensor(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Convert batch of token IDs to continuous representation tensor"""
        centers = torch.zeros_like(token_ids, dtype=torch.float32)
        for i in range(token_ids.shape[0]):
            for j in range(token_ids.shape[1]):
                centers[i, j] = self.token_to_center(token_ids[i, j].item())
        return centers

# Initialize token mapper
token_mapper = TokenIntervalMapper(VOCAB_SIZE)

# Memory management for Colab
def clear_memory():
    """Clear GPU memory cache"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print(f"Device: {device}")
print(f"Token mapper initialized with vocab size: {VOCAB_SIZE}")
print("Ready to run modified LLaDA algorithm with continuous token intervals!")



✅ Using GPU: Tesla T4
✅ GPU Memory: 15.8 GB
Device: cuda
Ready to run LLaDA algorithm!


## 2. Forward Process (Noise Addition)

The forward process in LLaDA adds noise by masking tokens according to a linear noise schedule. This is the core of the diffusion process.


In [ ]:
def forward_process_continuous(batch: torch.Tensor, prompt_index: torch.Tensor, 
                               token_mapper: TokenIntervalMapper, 
                               mask_value: float = 0.5) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Forward process for continuous LLaDA diffusion model.
    
    Instead of masking tokens, we add continuous noise to the token representations.
    The noise schedule determines how much noise to add to each position.

    Args:
        batch: Input token sequence of shape (batch_size, seq_len)
        prompt_index: Boolean mask indicating prompt positions
        token_mapper: TokenIntervalMapper instance
        mask_value: Value to use for "masked" positions (center of [0,1) interval)

    Returns:
        noisy_batch: Batch with continuous token representations + noise
        p_noise: Noise probability for each position
    """
    b, l = batch.shape
    
    # Convert discrete tokens to continuous representations
    continuous_batch = token_mapper.tokens_to_tensor(batch).to(batch.device)
    
    # Calculate target length (non-prompt tokens)
    target_len = (l - prompt_index.sum()).item()
    k = torch.randint(1, target_len + 1, (), device=batch.device)

    # Create linear noise schedule
    x = torch.round(torch.linspace(float(k), k + (b - 1) * (target_len / b), steps=b, device=batch.device)).long()
    x = ((x - 1) % target_len) + 1
    assert x.min() >= 1 and x.max() <= target_len

    # Create noise mask for target tokens
    indices = torch.arange(target_len, device=batch.device).repeat(b, 1)
    is_noise = indices < x.unsqueeze(1)

    # Randomly permute noise positions
    for i in range(b):
        is_noise[i] = is_noise[i][torch.randperm(target_len)]

    # Combine prompt and target noise masks
    is_noise = torch.cat((torch.zeros(b, prompt_index.sum(), dtype=torch.bool, device=batch.device), is_noise), dim=1)
    
    # Add noise to continuous representations
    # For "noisy" positions, replace with mask_value + small random noise
    noise_mask = torch.rand_like(continuous_batch) * 0.1  # Small random noise
    noisy_batch = torch.where(is_noise, mask_value + noise_mask, continuous_batch)

    # Return noise probability
    return noisy_batch, (x / target_len).unsqueeze(1).repeat(1, l)

def forward_process(batch: torch.Tensor, prompt_index: torch.Tensor, mask_id: int = MASK_TOKEN_ID) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Original forward process for LLaDA diffusion model (kept for reference).

    Args:
        batch: Input token sequence of shape (batch_size, seq_len)
        prompt_index: Boolean mask indicating prompt positions
        mask_id: Token ID for mask token

    Returns:
        noisy_batch: Batch with masked tokens
        p_mask: Mask probability for each position
    """
    b, l = batch.shape

    # Calculate target length (non-prompt tokens)
    target_len = (l - prompt_index.sum()).item()
    k = torch.randint(1, target_len + 1, (), device=batch.device)

    # Create linear noise schedule
    x = torch.round(torch.linspace(float(k), k + (b - 1) * (target_len / b), steps=b, device=batch.device)).long()
    x = ((x - 1) % target_len) + 1
    assert x.min() >= 1 and x.max() <= target_len

    # Create mask indices for target tokens
    indices = torch.arange(target_len, device=batch.device).repeat(b, 1)
    is_mask = indices < x.unsqueeze(1)

    # Randomly permute mask positions
    for i in range(b):
        is_mask[i] = is_mask[i][torch.randperm(target_len)]

    # Combine prompt and target masks
    is_mask = torch.cat((torch.zeros(b, prompt_index.sum(), dtype=torch.bool, device=batch.device), is_mask), dim=1)
    noisy_batch = torch.where(is_mask, mask_id, batch)

    # Return mask probability
    return noisy_batch, (x / target_len).unsqueeze(1).repeat(1, l)


## 3. Gumbel Noise for Sampling

LLaDA uses Gumbel noise for sampling categorical distributions during generation.


In [ ]:
def add_gumbel_noise(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """
    Add Gumbel noise for sampling categorical distributions.

    According to arXiv:2409.02908, for MDM, low-precision Gumbel Max improves
    perplexity score but reduces generation quality. Thus, we use float64.

    Args:
        logits: Model output logits
        temperature: Sampling temperature

    Returns:
        Logits with Gumbel noise applied
    """
    if temperature == 0:
        return logits

    logits = logits.to(torch.float64)
    noise = torch.rand_like(logits, dtype=torch.float64)
    gumbel_noise = (- torch.log(noise)) ** temperature
    return logits.exp() / gumbel_noise


## 4. Token Transfer Calculation

This function calculates how many tokens should be unmasked at each step during the reverse process.


In [ ]:
def get_num_transfer_tokens(mask_index: torch.Tensor, steps: int) -> torch.Tensor:
    """
    Calculate the number of tokens to transfer (unmask) at each step.

    In the reverse process, the interval [0, 1] is uniformly discretized into steps intervals.
    Furthermore, because LLaDA employs a linear noise schedule (as defined in Eq. (8)),
    the expected number of tokens transitioned at each step should be consistent.

    Args:
        mask_index: Boolean tensor indicating masked positions
        steps: Number of denoising steps

    Returns:
        Number of tokens to transfer at each step
    """
    mask_num = mask_index.sum(dim=1, keepdim=True)

    base = mask_num // steps
    remainder = mask_num % steps

    num_transfer_tokens = torch.zeros(mask_num.size(0), steps, device=mask_index.device, dtype=torch.int64) + base

    for i in range(mask_num.size(0)):
        num_transfer_tokens[i, :remainder[i]] += 1

    return num_transfer_tokens


## 5. Classifier-Free Guidance

Classifier-free guidance improves generation quality by using both conditional and unconditional predictions.


In [ ]:
def get_logits_with_cfg(model, batch: torch.Tensor, prompt_index: torch.Tensor, cfg_scale: float, mask_id: int = MASK_TOKEN_ID) -> torch.Tensor:
    """
    Get model logits with optional classifier-free guidance.

    Args:
        model: LLaDA model
        batch: Input batch
        prompt_index: Boolean mask for prompt positions
        cfg_scale: Classifier-free guidance scale
        mask_id: Mask token ID

    Returns:
        Guided logits
    """
    if cfg_scale > 0.:
        assert len(prompt_index) == batch.shape[1]
        prompt_index = prompt_index.unsqueeze(0).repeat(batch.shape[0], 1)
        un_batch = batch.clone()
        un_batch[prompt_index] = mask_id
        batch = torch.cat([batch, un_batch])

    logits = model(batch).logits

    if cfg_scale > 0.:
        logits, un_logits = torch.chunk(logits, 2, dim=0)
        logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
    return logits


## 6. Core Generation Algorithm

This is the main LLaDA generation algorithm that implements the reverse diffusion process with semi-autoregressive sampling and remasking strategies.


In [ ]:
@torch.no_grad()
def generate_continuous(model, prompt: torch.Tensor, token_mapper: TokenIntervalMapper,
                       steps: int = 128, gen_length: int = 128, block_length: int = 128, 
                       temperature: float = 0., cfg_scale: float = 0.,
                       remasking: str = 'low_confidence', mask_value: float = 0.5) -> torch.Tensor:
    """
    Generate text using continuous LLaDA diffusion model.

    Args:
        model: LLaDA model (continuous token predictor)
        prompt: Input prompt tensor of shape (1, L)
        token_mapper: TokenIntervalMapper instance
        steps: Number of sampling steps (≤ gen_length)
        gen_length: Generated answer length
        block_length: Block length for semi-autoregressive generation (≤ gen_length)
        temperature: Sampling temperature for continuous values
        cfg_scale: Unsupervised classifier-free guidance scale
        remasking: Remasking strategy ('low_confidence' or 'random')
        mask_value: Value to use for "masked" positions

    Returns:
        Generated sequence tensor (continuous values)
    """
    # Initialize sequence with continuous representations
    # Start with mask_value for generation positions
    x = torch.full((1, prompt.shape[1] + gen_length), mask_value, dtype=torch.float32).to(model.device)
    
    # Convert prompt tokens to continuous representations
    prompt_continuous = token_mapper.tokens_to_tensor(prompt).to(model.device)
    x[:, :prompt.shape[1]] = prompt_continuous.clone()

    prompt_index = torch.arange(prompt.shape[1], device=model.device)

    # Validate block configuration
    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length

    assert steps % num_blocks == 0
    steps_per_block = steps // num_blocks

    # Process each block
    for num_block in range(num_blocks):
        block_start = prompt.shape[1] + num_block * block_length
        block_end = prompt.shape[1] + (num_block + 1) * block_length

        # Get mask indices for current block (positions with mask_value)
        block_mask_index = torch.abs(x[:, block_start:block_end] - mask_value) < 1e-6
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps_per_block)

        # Process each step in the block
        for i in range(steps_per_block):
            mask_index = torch.abs(x - mask_value) < 1e-6

            # Get model predictions with optional CFG
            if cfg_scale > 0.:
                un_x = x.clone()
                un_x[prompt_index] = mask_value
                x_ = torch.cat([x, un_x], dim=0)
                logits = model(x_).logits
                logits, un_logits = torch.chunk(logits, 2, dim=0)
                logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
            else:
                logits = model(x).logits

            # Apply temperature scaling for continuous sampling
            if temperature > 0:
                logits = logits / temperature
            
            # Sample continuous values (assuming model outputs continuous predictions)
            x0 = torch.tanh(logits)  # Clamp to [-1, 1] range, then map to [0, 1]
            x0 = (x0 + 1) / 2  # Map from [-1, 1] to [0, 1]

            # Calculate confidence scores for remasking
            if remasking == 'low_confidence':
                # Use variance as confidence measure
                confidence = -torch.abs(logits)  # Lower absolute logits = lower confidence
            elif remasking == 'random':
                confidence = torch.rand((x0.shape[0], x0.shape[1]), device=x0.device)
            else:
                raise NotImplementedError(f"Remasking strategy '{remasking}' not implemented")

            # Don't consider positions beyond current block
            confidence[:, block_end:] = -np.inf

            # Apply predictions where we have masks
            x0 = torch.where(mask_index, x0, x)
            confidence = torch.where(mask_index, confidence, -np.inf)

            # Select tokens to unmask based on confidence
            transfer_index = torch.zeros_like(x0, dtype=torch.bool, device=x0.device)
            for j in range(confidence.shape[0]):
                _, select_index = torch.topk(confidence[j], k=num_transfer_tokens[j, i])
                transfer_index[j, select_index] = True

            # Apply the selected tokens
            x[transfer_index] = x0[transfer_index]

    return x

@torch.no_grad()
def generate(model, prompt: torch.Tensor, steps: int = 128, gen_length: int = 128,
             block_length: int = 128, temperature: float = 0., cfg_scale: float = 0.,
             remasking: str = 'low_confidence', mask_id: int = MASK_TOKEN_ID) -> torch.Tensor:
    """
    Original generate function for discrete LLaDA diffusion model (kept for reference).

    Args:
        model: LLaDA model (mask predictor)
        prompt: Input prompt tensor of shape (1, L)
        steps: Number of sampling steps (≤ gen_length)
        gen_length: Generated answer length
        block_length: Block length for semi-autoregressive generation (≤ gen_length)
        temperature: Categorical distribution sampling temperature
        cfg_scale: Unsupervised classifier-free guidance scale
        remasking: Remasking strategy ('low_confidence' or 'random')
        mask_id: Token ID of [MASK]

    Returns:
        Generated sequence tensor
    """
    # Initialize sequence with masks
    x = torch.full((1, prompt.shape[1] + gen_length), mask_id, dtype=torch.long).to(model.device)
    x[:, :prompt.shape[1]] = prompt.clone()

    prompt_index = (x != mask_id)

    # Validate block configuration
    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length

    assert steps % num_blocks == 0
    steps_per_block = steps // num_blocks

    # Process each block
    for num_block in range(num_blocks):
        block_start = prompt.shape[1] + num_block * block_length
        block_end = prompt.shape[1] + (num_block + 1) * block_length

        # Get mask indices for current block
        block_mask_index = (x[:, block_start:block_end] == mask_id)
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps_per_block)

        # Process each step in the block
        for i in range(steps_per_block):
            mask_index = (x == mask_id)

            # Get model predictions with optional CFG
            if cfg_scale > 0.:
                un_x = x.clone()
                un_x[prompt_index] = mask_id
                x_ = torch.cat([x, un_x], dim=0)
                logits = model(x_).logits
                logits, un_logits = torch.chunk(logits, 2, dim=0)
                logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
            else:
                logits = model(x).logits

            # Apply Gumbel noise for sampling
            logits_with_noise = add_gumbel_noise(logits, temperature=temperature)
            x0 = torch.argmax(logits_with_noise, dim=-1)

            # Calculate confidence scores for remasking
            if remasking == 'low_confidence':
                p = F.softmax(logits, dim=-1)
                x0_p = torch.squeeze(
                    torch.gather(p, dim=-1, index=torch.unsqueeze(x0, -1)), -1)
            elif remasking == 'random':
                x0_p = torch.rand((x0.shape[0], x0.shape[1]), device=x0.device)
            else:
                raise NotImplementedError(f"Remasking strategy '{remasking}' not implemented")

            # Don't consider positions beyond current block
            x0_p[:, block_end:] = -np.inf

            # Apply predictions where we have masks
            x0 = torch.where(mask_index, x0, x)
            confidence = torch.where(mask_index, x0_p, -np.inf)

            # Select tokens to unmask based on confidence
            transfer_index = torch.zeros_like(x0, dtype=torch.bool, device=x0.device)
            for j in range(confidence.shape[0]):
                _, select_index = torch.topk(confidence[j], k=num_transfer_tokens[j, i])
                transfer_index[j, select_index] = True

            # Apply the selected tokens
            x[transfer_index] = x0[transfer_index]

    return x


## 7. Log Likelihood Estimation

This function estimates the log likelihood of a sequence using Monte Carlo sampling, which is used for evaluation.


In [ ]:
@torch.no_grad()
def get_log_likelihood(model, prompt: torch.Tensor, answer: torch.Tensor,
                      mc_num: int = 128, batch_size: int = 16, cfg_scale: float = 0.,
                      mask_id: int = MASK_TOKEN_ID) -> float:
    """
    Estimate log likelihood using Monte Carlo sampling.

    Args:
        model: LLaDA model
        prompt: Prompt tensor of shape (l1)
        answer: Answer tensor of shape (l2)
        mc_num: Monte Carlo estimation times
        batch_size: Mini batch size
        cfg_scale: Unsupervised classifier-free guidance scale
        mask_id: Token ID of [MASK]

    Returns:
        Estimated log likelihood
    """
    seq = torch.concatenate([prompt, answer])[None, :]
    seq = seq.repeat((batch_size, 1)).to(model.device)
    prompt_index = torch.arange(seq.shape[1], device=model.device) < len(prompt)

    loss_ = []
    for _ in range(mc_num // batch_size):
        perturbed_seq, p_mask = forward_process(seq, prompt_index, mask_id)
        mask_index = perturbed_seq == mask_id

        logits = get_logits_with_cfg(model, perturbed_seq, prompt_index, cfg_scale, mask_id)

        loss = F.cross_entropy(logits[mask_index], seq[mask_index], reduction='none') / p_mask[mask_index]
        loss = loss.sum() / batch_size

        loss_.append(loss.item())

    return - sum(loss_) / len(loss_)


## 8. Training Loss Computation

This is the core training loss computation used during pre-training and supervised fine-tuning.


In [ ]:
def compute_training_loss_continuous(model, input_ids: torch.Tensor, token_mapper: TokenIntervalMapper,
                                    prompt_lengths: Optional[torch.Tensor] = None,
                                    eps: float = 1e-3, mask_value: float = 0.5) -> torch.Tensor:
    """
    Compute training loss for continuous LLaDA model.

    This function implements the core training objective for continuous token representations:
    - Forward process: Add continuous noise to token representations
    - Loss computation: MSE loss on noisy positions weighted by noise probability

    Args:
        model: LLaDA model
        input_ids: Input token sequence of shape (batch_size, seq_len)
        token_mapper: TokenIntervalMapper instance
        prompt_lengths: Optional prompt lengths for SFT (if None, treats as pre-training)
        eps: Minimum noise probability
        mask_value: Value to use for "masked" positions

    Returns:
        Training loss tensor
    """
    b, l = input_ids.shape

    # Convert discrete tokens to continuous representations
    continuous_input = token_mapper.tokens_to_tensor(input_ids).to(input_ids.device)

    # Forward process: add continuous noise
    t = torch.rand(b, device=input_ids.device)
    p_noise = (1 - eps) * t + eps
    p_noise = p_noise[:, None].repeat(1, l)

    # Add Gaussian noise to continuous representations
    noise = torch.randn_like(continuous_input) * 0.1  # Small Gaussian noise
    noise_mask = torch.rand((b, l), device=input_ids.device) < p_noise
    
    # For noisy positions, replace with mask_value + noise
    noisy_batch = torch.where(noise_mask, mask_value + noise, continuous_input)

    # For SFT: don't add noise to prompt tokens
    if prompt_lengths is not None:
        token_positions = torch.arange(noisy_batch.shape[1], device=noisy_batch.device).expand(noisy_batch.size(0), noisy_batch.size(1))
        prompt_mask = (token_positions < prompt_lengths.unsqueeze(1))
        noisy_batch[prompt_mask] = continuous_input[prompt_mask]
        noise_mask = torch.abs(noisy_batch - continuous_input) > 1e-6

    # Get model predictions (continuous values)
    logits = model(input_ids=noisy_batch).logits
    
    # Convert logits to [0, 1] range if needed
    if logits.min() < 0 or logits.max() > 1:
        logits = torch.sigmoid(logits)  # Map to [0, 1]

    # Compute MSE loss on noisy positions
    token_loss = F.mse_loss(logits[noise_mask], continuous_input[noise_mask], reduction='none') / p_noise[noise_mask]

    if prompt_lengths is not None:
        # For SFT: normalize by answer length
        prompt_mask = prompt_mask.to(torch.int64)
        answer_lengths = torch.sum((1 - prompt_mask), dim=-1, keepdim=True)
        answer_lengths = answer_lengths.repeat(1, noisy_batch.shape[1])
        loss = torch.sum(token_loss / answer_lengths[noise_mask]) / input_ids.shape[0]
    else:
        # For pre-training: normalize by sequence length
        loss = token_loss.sum() / (input_ids.shape[0] * input_ids.shape[1])

    return loss

def compute_training_loss(model, input_ids: torch.Tensor, prompt_lengths: Optional[torch.Tensor] = None,
                         eps: float = 1e-3, mask_id: int = MASK_TOKEN_ID) -> torch.Tensor:
    """
    Original training loss computation for discrete LLaDA model (kept for reference).

    This function implements the core training objective from the paper:
    - Forward process: Add noise by masking tokens
    - Loss computation: Cross-entropy on masked positions weighted by mask probability

    Args:
        model: LLaDA model
        input_ids: Input token sequence of shape (batch_size, seq_len)
        prompt_lengths: Optional prompt lengths for SFT (if None, treats as pre-training)
        eps: Minimum mask probability
        mask_id: Token ID for mask token

    Returns:
        Training loss tensor
    """
    b, l = input_ids.shape

    # Forward process: add noise by masking
    t = torch.rand(b, device=input_ids.device)
    p_mask = (1 - eps) * t + eps
    p_mask = p_mask[:, None].repeat(1, l)

    masked_indices = torch.rand((b, l), device=input_ids.device) < p_mask
    noisy_batch = torch.where(masked_indices, mask_id, input_ids)

    # For SFT: don't mask prompt tokens
    if prompt_lengths is not None:
        token_positions = torch.arange(noisy_batch.shape[1], device=noisy_batch.device).expand(noisy_batch.size(0), noisy_batch.size(1))
        prompt_mask = (token_positions < prompt_lengths.unsqueeze(1))
        noisy_batch[prompt_mask] = input_ids[prompt_mask]
        masked_indices = (noisy_batch == mask_id)

    # Get model predictions
    logits = model(input_ids=noisy_batch).logits

    # Compute weighted cross-entropy loss
    token_loss = F.cross_entropy(logits[masked_indices], input_ids[masked_indices], reduction='none') / p_mask[masked_indices]

    if prompt_lengths is not None:
        # For SFT: normalize by answer length
        prompt_mask = prompt_mask.to(torch.int64)
        answer_lengths = torch.sum((1 - prompt_mask), dim=-1, keepdim=True)
        answer_lengths = answer_lengths.repeat(1, noisy_batch.shape[1])
        loss = torch.sum(token_loss / answer_lengths[masked_indices]) / input_ids.shape[0]
    else:
        # For pre-training: normalize by sequence length
        loss = token_loss.sum() / (input_ids.shape[0] * input_ids.shape[1])

    return loss


## 9. Example Usage

Here's how to use the LLaDA algorithm with a pre-trained model:


In [ ]:
# Test the modified continuous LLaDA algorithm
def test_continuous_algorithm():
    """
    Test the modified LLaDA algorithm with continuous token intervals.
    This demonstrates the key differences from the original discrete approach.
    """
    print("🧪 Testing Continuous LLaDA Algorithm")
    print("=" * 50)
    
    # Test token mapping
    print("\n1. Testing Token-to-Interval Mapping:")
    test_tokens = torch.tensor([1, 100, 1000, VOCAB_SIZE-1])
    print(f"Test tokens: {test_tokens.tolist()}")
    
    for token_id in test_tokens:
        interval = token_mapper.token_to_interval(token_id.item())
        center = token_mapper.token_to_center(token_id.item())
        print(f"Token {token_id.item()}: interval [{interval[0]:.6f}, {interval[1]:.6f}), center: {center:.6f}")
    
    # Test forward process
    print("\n2. Testing Continuous Forward Process:")
    batch = torch.tensor([[1, 2, 3, 4, 5]], device=device)
    prompt_index = torch.tensor([True, True, False, False, False], device=device)
    
    print(f"Original batch: {batch}")
    print(f"Prompt index: {prompt_index}")
    
    noisy_batch, p_noise = forward_process_continuous(batch, prompt_index, token_mapper)
    print(f"Noisy batch (continuous): {noisy_batch}")
    print(f"Noise probabilities: {p_noise}")
    
    # Test reverse mapping
    print("\n3. Testing Continuous-to-Token Conversion:")
    continuous_values = torch.tensor([0.1, 0.5, 0.9])
    for val in continuous_values:
        token_id = token_mapper.interval_to_token(val.item())
        print(f"Continuous value {val.item():.3f} -> Token {token_id}")
    
    # Test training loss computation
    print("\n4. Testing Continuous Training Loss:")
    # Create a simple mock model for testing
    class MockModel:
        def __init__(self):
            self.device = device
        
        def __call__(self, input_ids):
            # Mock model that returns continuous values
            class MockOutput:
                def __init__(self, logits):
                    self.logits = logits
            return MockOutput(torch.randn_like(input_ids))
    
    mock_model = MockModel()
    test_input = torch.tensor([[1, 2, 3, 4, 5]], device=device)
    
    try:
        loss = compute_training_loss_continuous(mock_model, test_input, token_mapper)
        print(f"Training loss: {loss.item():.6f}")
    except Exception as e:
        print(f"Error in training loss computation: {e}")
    
    print("\n✅ Continuous LLaDA algorithm test completed!")
    print("\nKey Differences from Original:")
    print("- Tokens mapped to continuous intervals [0, 1)")
    print("- Forward process adds continuous noise instead of discrete masking")
    print("- Training uses MSE loss instead of cross-entropy")
    print("- Generation works with continuous values")

# Run the test
test_continuous_algorithm()

# Example usage with pre-trained LLaDA model
def example_usage():
    """
    Example of how to use the LLaDA algorithm with a pre-trained model.
    Note: This requires the transformers library and a pre-trained LLaDA model.
    """
    try:
        from transformers import AutoTokenizer, AutoModel

        # Load pre-trained model and tokenizer
        model_path = 'GSAI-ML/LLaDA-8B-Instruct'  # or 'GSAI-ML/LLaDA-8B-Base'

        print("Loading model and tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        model = AutoModel.from_pretrained(model_path, trust_remote_code=True, torch_dtype=torch.bfloat16).to(device).eval()

        # Example prompt
        prompt_text = "What is the capital of France?"

        # For Instruct model, use chat template
        messages = [{"role": "user", "content": prompt_text}]
        chat_input = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        input_ids = tokenizer(chat_input)['input_ids']
        input_ids = torch.tensor(input_ids).to(device).unsqueeze(0)

        print(f"Prompt: {prompt_text}")
        print(f"Input IDs shape: {input_ids.shape}")

        # Generate response using original discrete method
        print("Generating response with original discrete method...")
        generated = generate(
            model,
            input_ids,
            steps=128,
            gen_length=128,
            block_length=32,
            temperature=0.,
            cfg_scale=0.,
            remasking='low_confidence'
        )

        # Decode response
        response_tokens = generated[0, input_ids.shape[1]:]
        response_text = tokenizer.decode(response_tokens, skip_special_tokens=True)

        print(f"Generated response: {response_text}")

        return response_text

    except ImportError:
        print("transformers library not available. Please install it to run this example.")
        print("pip install transformers")
        return None
    except Exception as e:
        print(f"Error loading model: {e}")
        print("Make sure you have access to the LLaDA model and sufficient GPU memory.")
        return None

# Uncomment to run the example
# example_usage()


KeyboardInterrupt: 

## 10. Modified Algorithm Summary: Continuous Token Intervals

The modified LLaDA algorithm now uses **continuous token intervals** instead of discrete masking, creating a true diffusion process over continuous space.

### Key Modifications:

1. **Token-to-Interval Mapping**: Each discrete token ID is mapped to a unique interval in [0, 1), where:
   - Token 0 → [0, 1/vocab_size)
   - Token 1 → [1/vocab_size, 2/vocab_size)
   - Token k → [k/vocab_size, (k+1)/vocab_size)
   - All intervals are disjoint and cover [0, 1)

2. **Continuous Forward Process**: Instead of masking tokens with `[MASK]`, we:
   - Convert discrete tokens to continuous representations (interval centers)
   - Add Gaussian noise to continuous values
   - Use a linear noise schedule to determine noise intensity

3. **Continuous Reverse Process**: The generation process now:
   - Works with continuous values in [0, 1)
   - Uses temperature scaling for continuous sampling
   - Applies MSE loss instead of cross-entropy

4. **Training Loss**: Modified to use:
   - MSE loss between predicted and target continuous values
   - Weighted by noise probability (similar to original)
   - Continuous noise addition instead of discrete masking

### Algorithm Comparison:

| Aspect | Original LLaDA | Modified LLaDA |
|--------|----------------|----------------|
| **Token Representation** | Discrete IDs | Continuous intervals [0, 1) |
| **Forward Process** | Masking with `[MASK]` token | Adding Gaussian noise |
| **Reverse Process** | Token prediction | Continuous value prediction |
| **Training Loss** | Cross-entropy | MSE |
| **Sampling** | Gumbel noise + argmax | Temperature scaling + continuous sampling |
| **Noise Type** | Discrete masking | Continuous Gaussian noise |

### Benefits of Continuous Approach:

1. **True Diffusion**: Now implements actual diffusion over continuous space
2. **Smooth Interpolation**: Can represent intermediate states between tokens
3. **Gradient Flow**: Better gradient flow for continuous optimization
4. **Flexible Sampling**: More flexible sampling strategies possible

### Implementation Notes:

- **TokenIntervalMapper**: Handles bidirectional mapping between discrete tokens and continuous intervals
- **forward_process_continuous**: Adds continuous noise instead of discrete masking
- **generate_continuous**: Generates continuous values and converts back to tokens
- **compute_training_loss_continuous**: Uses MSE loss for continuous predictions

This modification transforms LLaDA from a masking-based iterative refinement model into a true diffusion model operating over continuous token representations.
